In [1]:
import os
import re
import time
import json
import random
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import tqdm
from transformers import EsmTokenizer, EsmForMaskedLM

In [2]:
# Replace with your antibody wild-type (WT) sequence
seq_name = "1mel"
wt_sequence = "VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREGVAAINMGGGITYYADSVKGRFTISQDNAKNTVYLLMNSLEPEDTAIYYCAADSTIYASYYECGHGLSTGGYGYDSWGQGTQVTVSS"

model_name = "facebook/esm2_t33_650M_UR50D"
print(f"Loading ESM-2 model: {model_name}...")
try:
    tokenizer = EsmTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_name)
    model.eval()
    print("Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# Parameters
num_variants = 100000
mutation_counts = [2]  # number of mutations per variant
calculate_pll = True   # set True to compute PLL (slower)
filter_top_pll = True  # set True to output only top PLL %
top_pll_percent = 1    # top percentile to keep (0.1-100.0)
output_dir = "."  # current directory (where the notebook is)
output_csv_filename_base = f"multi_mutation_esm2_650M_{mutation_counts[0]}muts_{seq_name}_{num_variants}"

print(f"WT sequence: {wt_sequence[:30]}... (Length: {len(wt_sequence)})")
print(f"Target variants: {num_variants}")
print(f"Mutation counts: {mutation_counts}\n")

Loading ESM-2 model: facebook/esm2_t33_650M_UR50D...
Model and tokenizer loaded successfully.
WT sequence: VQLQASGGGSVQAGGSLRLSCAASGYTIGP... (Length: 132)
Target variants: 100000
Mutation counts: [2]



In [3]:
# Check if a GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model.to(device)
model.eval()

def generate_large_scale_variants(wt_seq, num_variants=10000, mutation_counts=[2, 3, 4, 5]):
    """
    Generate many variants using ESM-2 probability distribution (zero-shot).
    """
    print(f"Processing WT sequence (Length: {len(wt_seq)})...")
    
    # 1. Compute probability matrix with model (on GPU)
    inputs = tokenizer(wt_seq, return_tensors="pt", add_special_tokens=False).to(device)
    input_ids = inputs["input_ids"]  # shape: [1, seq_len]
    seq_len = input_ids.shape[1]
    
    with torch.no_grad():
        logits = model(input_ids).logits  # [1, seq_len, vocab_size]
        probs = F.softmax(logits, dim=-1)[0]  # [seq_len, vocab_size]
    
    # 2. Move probability matrix to CPU
    probs = probs.cpu().numpy()
    wt_ids = input_ids[0].cpu().numpy()
    
    # Build mutation candidate list
    mutation_candidates = []
    weights = []
    
    valid_aa_indices = []
    # Valid amino acid indices (exclude special tokens)
    for i in range(tokenizer.vocab_size):
        char = tokenizer.decode([i])
        if char not in ["<cls>", "<pad>", "<eos>", "<unk>", "-", "X", "B", "U", "Z", "O"]:
            valid_aa_indices.append(i)
            
    print("Building mutation probability map...")
    
    for pos in range(seq_len):
        wt_aa_id = wt_ids[pos]
        
        for aa_id in valid_aa_indices:
            if aa_id == wt_aa_id:
                continue
            
            mut_prob = probs[pos, aa_id]
            
            if mut_prob < 0.001: 
                continue

            # (position, mutated AA)
            mutation_candidates.append((pos, tokenizer.decode([aa_id])))
            
            # Use probability as weight
            weights.append(mut_prob)
            
    # Normalize weights
    weights = np.array(weights)
    weights = weights / weights.sum()
    
    print(f"Total valid single mutations found: {len(mutation_candidates)}")
    print(f"Generating {num_variants} variants...")
    
    generated_variants = set()
    results = []
    
    # 3. Sample variants
    while len(results) < num_variants:
        # Randomly choose number of mutations
        n_muts = random.choice(mutation_counts)
        
        # Sample mutations by weight
        chosen_indices = np.random.choice(len(mutation_candidates), size=n_muts*2, replace=False, p=weights)
        
        current_mutations = []
        used_positions = set()
        
        for idx in chosen_indices:
            pos, aa = mutation_candidates[idx]
            if pos not in used_positions:
                current_mutations.append((pos, aa))
                used_positions.add(pos)
            
            if len(current_mutations) == n_muts:
                break
        
        if len(current_mutations) < n_muts:
            continue
            
        # Sort for deduplication
        current_mutations.sort(key=lambda x: x[0])
        variant_id = tuple(current_mutations)
        
        if variant_id in generated_variants:
            continue
            
        generated_variants.add(variant_id)
        
        # Build sequence
        seq_list = list(wt_seq)
        mut_info_list = []
        for pos, aa in current_mutations:
            wt_aa = wt_seq[pos]
            seq_list[pos] = aa
            mut_info_list.append({
                "position": pos + 1,  # 1-based indexing
                "original_aa": wt_aa,
                "mutated_aa": aa
            })
            
        results.append({
            "sequence": "".join(seq_list),
            "mutations": mut_info_list,
            "num_mutations": n_muts
        })
        
        if len(results) % 1000 == 0:
            print(f"Generated {len(results)} variants...")

    return results

# Generate variants
variants = generate_large_scale_variants(wt_sequence, num_variants=num_variants, mutation_counts=mutation_counts)

print(f"\\nGeneration complete. Generated {len(variants)} variants.\\n")

# Show first 5 variants
print("--- Example Generated Variants ---")
for v in variants[:5]:
    muts_str = ", ".join([f"{m['original_aa']}{m['position']}{m['mutated_aa']}" for m in v['mutations']])
    print(f"[{v['num_mutations']} muts] {muts_str}")

# Mutation position summary
print("\n--- Mutation Position Analysis ---")
used_positions = set()
for variant in variants:
    for m in variant["mutations"]:
        used_positions.add(m["position"] - 1)  # 0-based index

all_positions = set(range(len(wt_sequence)))
unused_positions = sorted(all_positions - used_positions)

print(f"Total sequence length: {len(wt_sequence)}")
print(f"Positions with mutations: {len(used_positions)}")
print(f"Positions without mutations: {len(unused_positions)}")
if unused_positions:
    # Show as 1-based
    unused_positions_1based = [pos + 1 for pos in unused_positions]
    print(f"Unused positions (1-based): {unused_positions_1based}")
    print(f"Unused positions (0-based): {unused_positions}")

# PLL (pseudo log-likelihood) calculation (optional, batched)
def calc_pseudo_pll_batch(variants, model, tokenizer, wt_sequence):
    """
    Compute PLL for a list of variants in batch.
    Caches log-probs per position for efficiency.
    
    Args:
        variants: list of dicts with "sequence", "mutations", etc.
        model: ESM-2 model
        tokenizer: ESM-2 tokenizer
        wt_sequence: wild-type sequence
    Returns:
        pll_values: list of PLL per variant
    """
    model.eval()
    
    # 1. Collect all mutation positions
    all_mutation_positions = set()
    for variant in variants:
        mutations = variant["mutations"]
        for m in mutations:
            all_mutation_positions.add(m["position"] - 1)  # 1-based to 0-based
    
    all_mutation_positions = sorted(all_mutation_positions)
    print(f"Computing log probabilities for {len(all_mutation_positions)} unique mutation positions...")
    
    # 2. Precompute log-probs per position (cache)
    position_log_probs_cache = {}
    
    with torch.no_grad():
        for pos in tqdm.tqdm(all_mutation_positions, desc="Precomputing log probs"):
            # Mask this position
            sequence_list = list(wt_sequence)
            sequence_list[pos] = tokenizer.mask_token
            masked_sequence = "".join(sequence_list)
            
            # Tokenize
            inputs = tokenizer(masked_sequence, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get mask index
            masked_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1][0]
            
            # Forward pass
            outputs = model(**inputs)
            logits = outputs.logits[0, masked_index, :]
            log_probs = F.log_softmax(logits, dim=-1)
            
            # Cache on CPU
            position_log_probs_cache[pos] = log_probs.cpu()
    
    # 3. Compute PLL per variant using cache
    print(f"Calculating PLL for {len(variants)} variants...")
    pll_values = []
    
    for variant in tqdm.tqdm(variants, desc="Calculating PLL"):
        mutations = variant["mutations"]
        
        if len(mutations) == 0:
            pll_values.append(float("-inf"))
            continue
        
        total_log_likelihood = 0.0
        for m in mutations:
            pos = m["position"] - 1  # 1-based to 0-based
            mutated_aa = m["mutated_aa"]
            mutated_token_id = tokenizer.convert_tokens_to_ids(mutated_aa)
            
            # Get log-probs from cache
            log_probs = position_log_probs_cache[pos]
            total_log_likelihood += log_probs[mutated_token_id].item()
        
        # PLL = mean log-likelihood over mutation sites
        pll = total_log_likelihood / len(mutations)
        pll_values.append(pll)
    
    return pll_values

# Compute PLL (optional)
if calculate_pll:
    print(f"\\nCalculating PLL (mutations only) for {len(variants)} variants using batch processing...")
    pll_start_time = time.time()
    
    # Compute PLL in batch
    pll_values = calc_pseudo_pll_batch(variants, model, tokenizer, wt_sequence)
    
    # Attach PLL to each variant
    for i, variant in enumerate(variants):
        variant["pll"] = pll_values[i]
    
    pll_time = time.time() - pll_start_time
    print(f"PLL calculation finished in {pll_time:.2f} seconds.")
else:
    print("\\nPLL calculation skipped (calculate_pll = False)")

# Build DataFrame and save CSV
df_data = []
for variant in variants:
    row_data = {
        "sequence": variant["sequence"],
        "num_mutations": variant["num_mutations"],
        "mutations": json.dumps(variant["mutations"]),
    }
    # Add PLL column if computed
    if calculate_pll:
        row_data["pll"] = variant["pll"]
    df_data.append(row_data)

df_output = pd.DataFrame(df_data)

# Filter to top PLL % (optional)
if filter_top_pll and calculate_pll:
    if "pll" not in df_output.columns:
        print("\\nWarning: PLL column not found. Skipping filtering.")
    else:
        # Sort by PLL descending
        df_output = df_output.sort_values("pll", ascending=False)
        
        # Take top X%
        num_top = max(1, int(len(df_output) * (top_pll_percent / 100.0)))
        df_output = df_output.head(num_top).copy()
        
        print(f"\\nFiltered to top {top_pll_percent}% (PLL): {len(df_output)} variants")
        if len(df_output) > 0:
            print(f"PLL range: {df_output['pll'].min():.4f} to {df_output['pll'].max():.4f}")
elif filter_top_pll and not calculate_pll:
    print("\\nWarning: filter_top_pll is True but calculate_pll is False. Skipping filtering.")

# Set output filename
if filter_top_pll and calculate_pll and "pll" in df_output.columns:
    # Add suffix when filtered
    output_csv_filename = f"{output_csv_filename_base}_top{top_pll_percent}pct.csv"
else:
    output_csv_filename = f"{output_csv_filename_base}.csv"

# Save CSV
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, output_csv_filename)
df_output.to_csv(output_path, index=False)

print(f"\\nSuccessfully saved {len(df_output)} sequences to '{output_path}'")
print(f"\\nColumns: {list(df_output.columns)}")
print(f"\\nFirst few rows:")
print(df_output.head())

Using device: cuda
Processing WT sequence (Length: 132)...


2026-02-17 13:49:17.412413: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-17 13:49:17.412448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-17 13:49:17.413280: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-17 13:49:18.116214: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Building mutation probability map...
Total valid single mutations found: 1205
Generating 100000 variants...
Generated 1000 variants...
Generated 2000 variants...
Generated 3000 variants...
Generated 4000 variants...
Generated 5000 variants...
Generated 6000 variants...
Generated 7000 variants...
Generated 8000 variants...
Generated 9000 variants...
Generated 10000 variants...
Generated 11000 variants...
Generated 12000 variants...
Generated 13000 variants...
Generated 14000 variants...
Generated 15000 variants...
Generated 16000 variants...
Generated 17000 variants...
Generated 18000 variants...
Generated 19000 variants...
Generated 20000 variants...
Generated 21000 variants...
Generated 22000 variants...
Generated 23000 variants...
Generated 24000 variants...
Generated 25000 variants...
Generated 26000 variants...
Generated 27000 variants...
Generated 28000 variants...
Generated 29000 variants...
Generated 30000 variants...
Generated 31000 variants...
Generated 32000 variants...
Gener

Precomputing log probs: 100%|██████████| 124/124 [00:04<00:00, 30.22it/s]


Calculating PLL for 100000 variants...


Calculating PLL: 100%|██████████| 100000/100000 [00:00<00:00, 190188.80it/s]


PLL calculation finished in 4.68 seconds.
\nFiltered to top 1% (PLL): 1000 variants
PLL range: -0.9467 to -0.0521
\nSuccessfully saved 1000 sequences to 'outputs/multi_mutation_esm2_650M_2muts_1mel_100000_top1pct.csv'
\nColumns: ['sequence', 'num_mutations', 'mutations', 'pll']
\nFirst few rows:
                                               sequence  num_mutations  \
1342  VQLQASGGGSVQPGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREW...              2   
2041  VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREW...              2   
2931  VQLQASGGGSVQPGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREG...              2   
64    VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREW...              2   
7013  VQLQASGGGSVQPGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREG...              2   

                                              mutations       pll  
1342  [{"position": 13, "original_aa": "A", "mutated... -0.052113  
2041  [{"position": 46, "original_aa": "G", "mutated... -0.081810  
2931  [{"position": 13, "original_aa": "A", "mutated..